# Phase 6: Engagement Prediction

This notebook implements the engagement prediction models as specified in the frozen guide.

In [1]:
import pandas as pd
import numpy as np
import os
import json
import yaml
import time
import joblib
from dotenv import find_dotenv
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, confusion_matrix
import xgboost as xgb
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.dirname(find_dotenv())
with open(os.path.join(PROJECT_ROOT, 'config.yaml'), 'r') as f:
    config = yaml.safe_load(f)

seed = config['seed']
train_rows = config['train_rows']

print(f"[Phase 6] Config loaded. Seed: {seed}, train_rows: {train_rows}")


[Phase 6] Config loaded. Seed: 42, train_rows: 400000


In [2]:
print("[Phase 6] Loading features...")
t0 = time.time()
cold_df = pd.read_parquet(os.path.join(PROJECT_ROOT, 'data/processed/features_cold.parquet'))
full_df = pd.read_parquet(os.path.join(PROJECT_ROOT, 'data/processed/features_full.parquet'))
print(f"Loaded cold features: {len(cold_df)} rows, full features: {len(full_df)} rows in {time.time()-t0:.2f}s")


[Phase 6] Loading features...


Loaded cold features: 1313584 rows, full features: 1313584 rows in 0.86s


In [3]:
print("[Phase 6] Validating schema & running leakage checks...")
cold_allowlist = config['feature_allowlist']['cold']
full_allowlist = config['feature_allowlist']['full']

assert set(cold_allowlist).issubset(cold_df.columns), "Missing cold features"
assert set(full_allowlist).issubset(full_df.columns), "Missing full features"

# Explicit leakage audit
# Check target is not in allowlist
for f in cold_allowlist + full_allowlist:
    assert f not in ['y', 'engagement_rate', 'likes', 'comments'], f"Leakage: {f} in allowlist"

print("Leakage audit passed.")


[Phase 6] Validating schema & running leakage checks...
Leakage audit passed.


In [4]:
print("[Phase 6] Preparing account-holdout split and time split...")
def get_xy(df, split_col, split_val, features, subsample=None):
    mask = df[split_col] == split_val
    sub = df[mask].copy()
    if subsample is not None:
        sub = sub.sample(n=subsample, random_state=seed)
    X = sub[features].copy()
    y = sub['y'].copy()
    er = sub['engagement_rate'].copy()
    return X, y, er, sub

# Primary splits (account holdout)
X_train_c, y_train_c, er_train_c, df_train_c = get_xy(cold_df, 'split', 'train', cold_allowlist, subsample=train_rows)
X_val_c, y_val_c, er_val_c, df_val_c = get_xy(cold_df, 'split', 'val', cold_allowlist)
X_test_c, y_test_c, er_test_c, df_test_c = get_xy(cold_df, 'split', 'test', cold_allowlist)

X_train_f, y_train_f, er_train_f, df_train_f = get_xy(full_df, 'split', 'train', full_allowlist, subsample=train_rows)
X_val_f, y_val_f, er_val_f, df_val_f = get_xy(full_df, 'split', 'val', full_allowlist)
X_test_f, y_test_f, er_test_f, df_test_f = get_xy(full_df, 'split', 'test', full_allowlist)

# Robustness splits (time split)
X_train_t, y_train_t, er_train_t, df_train_t = get_xy(cold_df, 'time_split', 'train', cold_allowlist, subsample=train_rows)
X_val_t, y_val_t, er_val_t, df_val_t = get_xy(cold_df, 'time_split', 'val', cold_allowlist)
X_test_t, y_test_t, er_test_t, df_test_t = get_xy(cold_df, 'time_split', 'test', cold_allowlist)

print("Data preparation complete.")


[Phase 6] Preparing account-holdout split and time split...


Data preparation complete.


In [5]:
def eval_metrics(y_true, y_pred, er_true):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    er_pred = (np.expm1(y_pred))/100.0
    er_pred = np.maximum(0, er_pred)
    mae_orig = mean_absolute_error(er_true, er_pred)
    
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAE_orig': mae_orig}

results = []


In [6]:
print("[Phase 6] Training baselines...")
# 1. Global median
global_med_y = y_train_c.median()
y_pred_val_gb = np.full_like(y_val_c, global_med_y)
res_gb = eval_metrics(y_val_c, y_pred_val_gb, er_val_c)
results.append({'feature_set':'cold', 'split':'account', 'model':'Global Median Baseline', 
                'val_RMSE':res_gb['RMSE'], 'val_MAE':res_gb['MAE'], 'val_R2':res_gb['R2'], 'val_MAE_orig':res_gb['MAE_orig']})

# 2. Category + Media Type median
train_group_medians = df_train_c.groupby(['original_category', 'media_type_carousel'])['y'].median().to_dict()
def predict_group_median(row):
    return train_group_medians.get((row['original_category'], row['media_type_carousel']), global_med_y)

y_pred_val_cb = df_val_c.apply(predict_group_median, axis=1)
res_cb = eval_metrics(y_val_c, y_pred_val_cb, er_val_c)
results.append({'feature_set':'cold', 'split':'account', 'model':'Category+Media Median Baseline', 
                'val_RMSE':res_cb['RMSE'], 'val_MAE':res_cb['MAE'], 'val_R2':res_cb['R2'], 'val_MAE_orig':res_cb['MAE_orig']})


[Phase 6] Training baselines...


In [7]:
print("[Phase 6] Training Linear Regression...")
t0 = time.time()
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])
lr.fit(X_train_c, y_train_c)
y_pred_val_lr = lr.predict(X_val_c)
res_lr = eval_metrics(y_val_c, y_pred_val_lr, er_val_c)
results.append({'feature_set':'cold', 'split':'account', 'model':'Linear Regression', 
                'val_RMSE':res_lr['RMSE'], 'val_MAE':res_lr['MAE'], 'val_R2':res_lr['R2'], 'val_MAE_orig':res_lr['MAE_orig']})
print(f"Linear Regression complete in {time.time()-t0:.2f}s")


[Phase 6] Training Linear Regression...


Linear Regression complete in 0.76s


In [8]:
print("[Phase 6] Tuning Random Forest...")
t0 = time.time()
rf_param_grid = {
    'min_samples_leaf': [1, 5, 10, 20, 50],
    'max_features': [0.3, 0.5, 0.7, 1.0, 'sqrt', 'log2']
}
rf = RandomForestRegressor(random_state=seed, n_estimators=100, n_jobs=-1)

# Guide says: Use validation data only for tuning/selection.
X_tune = pd.concat([X_train_c, X_val_c])
y_tune = pd.concat([y_train_c, y_val_c])
test_fold = np.concatenate([np.full(len(X_train_c), -1), np.full(len(X_val_c), 0)])
ps = PredefinedSplit(test_fold)

rf_search = RandomizedSearchCV(rf, rf_param_grid, n_iter=20, scoring='neg_root_mean_squared_error', 
                               cv=ps, random_state=seed, verbose=1, n_jobs=-1)
rf_search.fit(X_tune, y_tune)

best_rf = rf_search.best_estimator_
y_pred_val_rf = best_rf.predict(X_val_c)
res_rf = eval_metrics(y_val_c, y_pred_val_rf, er_val_c)
results.append({'feature_set':'cold', 'split':'account', 'model':'Random Forest', 
                'val_RMSE':res_rf['RMSE'], 'val_MAE':res_rf['MAE'], 'val_R2':res_rf['R2'], 'val_MAE_orig':res_rf['MAE_orig']})
print(f"Random Forest complete in {time.time()-t0:.2f}s. Best params: {rf_search.best_params_}")


[Phase 6] Tuning Random Forest...
Fitting 1 folds for each of 20 candidates, totalling 20 fits


Random Forest complete in 323.31s. Best params: {'min_samples_leaf': 50, 'max_features': 0.3}


In [9]:
print("[Phase 6] Training XGBoost...")
t0 = time.time()
xgb_model = xgb.XGBRegressor(tree_method="hist", random_state=seed, n_estimators=1000, early_stopping_rounds=50, learning_rate=0.05, max_depth=6)
# early stopping using validation data
xgb_model.fit(X_train_c, y_train_c, eval_set=[(X_val_c, y_val_c)], verbose=False)

y_pred_val_xgb = xgb_model.predict(X_val_c)
res_xgb = eval_metrics(y_val_c, y_pred_val_xgb, er_val_c)
results.append({'feature_set':'cold', 'split':'account', 'model':'XGBoost', 
                'val_RMSE':res_xgb['RMSE'], 'val_MAE':res_xgb['MAE'], 'val_R2':res_xgb['R2'], 'val_MAE_orig':res_xgb['MAE_orig']})
print(f"XGBoost complete in {time.time()-t0:.2f}s. Best iteration: {xgb_model.best_iteration}")


[Phase 6] Training XGBoost...


XGBoost complete in 2.37s. Best iteration: 148


In [10]:
print("[Phase 6] Training models for full feature set (reference)...")
# Full features - XGBoost
xgb_full = xgb.XGBRegressor(tree_method="hist", random_state=seed, n_estimators=1000, early_stopping_rounds=50, learning_rate=0.05, max_depth=6)
xgb_full.fit(X_train_f, y_train_f, eval_set=[(X_val_f, y_val_f)], verbose=False)
y_pred_val_xgb_f = xgb_full.predict(X_val_f)
res_xgb_f = eval_metrics(y_val_f, y_pred_val_xgb_f, er_val_f)
results.append({'feature_set':'full', 'split':'account', 'model':'XGBoost', 
                'val_RMSE':res_xgb_f['RMSE'], 'val_MAE':res_xgb_f['MAE'], 'val_R2':res_xgb_f['R2'], 'val_MAE_orig':res_xgb_f['MAE_orig']})

print("[Phase 6] Supplemental XGBoost on ALL training data (cold-start)...")
# Supplemental - all train rows
X_train_all, y_train_all, er_train_all, df_train_all = get_xy(cold_df, 'split', 'train', cold_allowlist) # no subsample
xgb_supp = xgb.XGBRegressor(tree_method="hist", random_state=seed, n_estimators=1000, early_stopping_rounds=50, learning_rate=0.05, max_depth=6)
xgb_supp.fit(X_train_all, y_train_all, eval_set=[(X_val_c, y_val_c)], verbose=False)
y_pred_val_xgb_supp = xgb_supp.predict(X_val_c)
res_xgb_supp = eval_metrics(y_val_c, y_pred_val_xgb_supp, er_val_c)
results.append({'feature_set':'cold', 'split':'account', 'model':'XGBoost (Supplemental All Train Rows)', 
                'val_RMSE':res_xgb_supp['RMSE'], 'val_MAE':res_xgb_supp['MAE'], 'val_R2':res_xgb_supp['R2'], 'val_MAE_orig':res_xgb_supp['MAE_orig']})

print("[Phase 6] Evaluating models... Robustness Time Split")
# Time Split - XGBoost
xgb_time = xgb.XGBRegressor(tree_method="hist", random_state=seed, n_estimators=1000, early_stopping_rounds=50, learning_rate=0.05, max_depth=6)
xgb_time.fit(X_train_t, y_train_t, eval_set=[(X_val_t, y_val_t)], verbose=False)
y_pred_val_xgb_t = xgb_time.predict(X_val_t)
res_xgb_t = eval_metrics(y_val_t, y_pred_val_xgb_t, er_val_t)
results.append({'feature_set':'cold', 'split':'time', 'model':'XGBoost', 
                'val_RMSE':res_xgb_t['RMSE'], 'val_MAE':res_xgb_t['MAE'], 'val_R2':res_xgb_t['R2'], 'val_MAE_orig':res_xgb_t['MAE_orig']})


[Phase 6] Training models for full feature set (reference)...


[Phase 6] Supplemental XGBoost on ALL training data (cold-start)...


[Phase 6] Evaluating models... Robustness Time Split


In [11]:
print("[Phase 6] Selecting model...")
df_res = pd.DataFrame(results)

# Primary comparison: feature_set='cold', split='account', excluding Supplemental
primary_res = df_res[(df_res['feature_set'] == 'cold') & (df_res['split'] == 'account') & (~df_res['model'].str.contains('Supplemental'))]
best_rmse = primary_res['val_RMSE'].min()
rf_rmse = primary_res[primary_res['model'] == 'Random Forest']['val_RMSE'].iloc[0]

if rf_rmse <= best_rmse * 1.01:
    selected_model_name = 'Random Forest'
    selected_model = best_rf
else:
    best_model_name = primary_res.loc[primary_res['val_RMSE'].idxmin(), 'model']
    selected_model_name = best_model_name
    if best_model_name == 'XGBoost':
        selected_model = xgb_model
    elif best_model_name == 'Linear Regression':
        selected_model = lr
    else:
        selected_model = None

print(f"Selected Model: {selected_model_name}")

print("[Phase 6] Scoring on test exactly once...")
y_pred_test = selected_model.predict(X_test_c)
test_metrics = eval_metrics(y_test_c, y_pred_test, er_test_c)

df_res['test_RMSE'] = np.nan
df_res['test_MAE'] = np.nan
df_res['test_R2'] = np.nan
df_res['test_MAE_orig'] = np.nan

for idx, row in df_res.iterrows():
    if row['model'] == selected_model_name and row['feature_set'] == 'cold' and row['split'] == 'account':
        df_res.loc[idx, 'test_RMSE'] = test_metrics['RMSE']
        df_res.loc[idx, 'test_MAE'] = test_metrics['MAE']
        df_res.loc[idx, 'test_R2'] = test_metrics['R2']
        df_res.loc[idx, 'test_MAE_orig'] = test_metrics['MAE_orig']

display(df_res)


[Phase 6] Selecting model...
Selected Model: Random Forest
[Phase 6] Scoring on test exactly once...


,feature_set,split,model,val_RMSE,val_MAE,val_R2,val_MAE_orig,test_RMSE,test_MAE,test_R2,test_MAE_orig
0,cold,account,Global Median Baseline,0.670025,0.539797,-0.002042,0.027537,NaN,NaN,NaN,NaN
1,cold,account,Category+Media Median Baseline,0.655788,0.526318,0.040090,0.027019,NaN,NaN,NaN,NaN
2,cold,account,Linear Regression,0.623270,0.499337,0.132927,0.025726,NaN,NaN,NaN,NaN
3,cold,account,Random Forest,0.562033,0.447175,0.294938,0.023375,0.606374,0.482893,0.193189,0.025901
4,cold,account,XGBoost,0.599324,0.477656,0.198273,0.024751,NaN,NaN,NaN,NaN
5,full,account,XGBoost,0.381751,0.284011,0.674716,0.016062,NaN,NaN,NaN,NaN
6,cold,account,XGBoost (Supplemental All Train Rows),0.599036,0.477575,0.199043,0.024742,NaN,NaN,NaN,NaN
7,cold,time,XGBoost,0.659969,0.522679,0.208549,0.029508,NaN,NaN,NaN,NaN


In [12]:
print("[Phase 6] Detailed breakdowns for selected model on Validation...")
y_pred_val_sel = selected_model.predict(X_val_c)
df_val_eval = df_val_c.copy()
df_val_eval['y_pred'] = y_pred_val_sel

print("1. Per-category breakdown (Val RMSE):")
for cat in df_val_eval['original_category'].unique():
    mask = df_val_eval['original_category'] == cat
    cat_rmse = np.sqrt(mean_squared_error(df_val_eval.loc[mask, 'y'], df_val_eval.loc[mask, 'y_pred']))
    print(f"  {cat}: {cat_rmse:.4f}")

print("\n2. Per-follower-tier breakdown (Val RMSE):")
df_val_eval['follower_tier'] = pd.qcut(df_val_eval['log_followers'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
for tier in ['Q1', 'Q2', 'Q3', 'Q4']:
    mask = df_val_eval['follower_tier'] == tier
    tier_rmse = np.sqrt(mean_squared_error(df_val_eval.loc[mask, 'y'], df_val_eval.loc[mask, 'y_pred']))
    print(f"  {tier}: {tier_rmse:.4f}")

print("\n3. Permutation Importance (Validation):")
result = permutation_importance(selected_model, X_val_c, y_val_c, n_repeats=5, random_state=seed, scoring='neg_root_mean_squared_error')
imp_df = pd.DataFrame({'feature': X_val_c.columns, 'importance': result.importances_mean})
imp_df = imp_df.sort_values('importance', ascending=False)
display(imp_df.head(10))


[Phase 6] Detailed breakdowns for selected model on Validation...
1. Per-category breakdown (Val RMSE):
  other: 0.6084
  beauty: 0.6093
  family: 0.5971
  fashion: 0.5534
  pet: 0.5278
  food: 0.4929
  interior: 0.5045
  travel: 0.5444
  fitness: 0.5646

2. Per-follower-tier breakdown (Val RMSE):
  Q1: 0.5668
  Q2: 0.5763


  Q3: 0.5511
  Q4: 0.5535

3. Permutation Importance (Validation):


,feature,importance
0,log_followers,0.119967
6,usertag_count,0.044223
4,hashtag_count,0.030883
14,category_other,0.026617
10,category_fashion,0.019550
5,caption_length,0.018966
12,category_food,0.009982
1,hour_of_day,0.009075
16,category_travel,0.005528
9,category_family,0.003880


In [13]:
print("[Phase 6] Low / Medium / High Labels...")
# Calculate thresholds on ALL TRAINING data within each pool
train_all_df = cold_df[cold_df['split'] == 'train']
pools = train_all_df['original_category'].unique().tolist() + ['all']
thresholds = {}

for pool in pools:
    if pool == 'all':
        er_pool = train_all_df['engagement_rate']
    else:
        er_pool = train_all_df[train_all_df['original_category'] == pool]['engagement_rate']
        
    p33 = np.percentile(er_pool, 33.33)
    p66 = np.percentile(er_pool, 66.67)
    thresholds[pool] = {'low': float(p33), 'high': float(p66)}

def apply_label(er, pool):
    t = thresholds.get(pool, thresholds['all'])
    if er < t['low']: return 'Low'
    elif er < t['high']: return 'Medium'
    else: return 'High'

df_val_eval['true_label'] = df_val_eval.apply(lambda row: apply_label(row['engagement_rate'], row['original_category']), axis=1)

er_pred_val = np.expm1(y_pred_val_sel) / 100.0
er_pred_val = np.maximum(0, er_pred_val)
df_val_eval['er_pred'] = er_pred_val
df_val_eval['pred_label'] = df_val_eval.apply(lambda row: apply_label(row['er_pred'], row['original_category']), axis=1)

acc = accuracy_score(df_val_eval['true_label'], df_val_eval['pred_label'])
print(f"3-class Accuracy: {acc:.4f}")

print("Confusion Matrix (Labels: High, Low, Medium):")
labels_order = ['Low', 'Medium', 'High']
conf_mat = confusion_matrix(df_val_eval['true_label'], df_val_eval['pred_label'], labels=labels_order)
conf_df = pd.DataFrame(conf_mat, index=[f"True_{l}" for l in labels_order], columns=[f"Pred_{l}" for l in labels_order])
display(conf_df)

majority_class = df_val_eval['true_label'].mode()[0]
maj_acc = (df_val_eval['true_label'] == majority_class).mean()
print(f"Majority-class baseline accuracy: {maj_acc:.4f}")


[Phase 6] Low / Medium / High Labels...


3-class Accuracy: 0.4484
Confusion Matrix (Labels: High, Low, Medium):


,Pred_Low,Pred_Medium,Pred_High
True_Low,6955,35381,1198
True_Medium,1893,37168,5162
True_High,546,28163,14689


Majority-class baseline accuracy: 0.3372


In [14]:
print("[Phase 6] Saving artifacts...")
os.makedirs(os.path.join(PROJECT_ROOT, 'artifacts/v1.0'), exist_ok=True)

joblib.dump({'model': selected_model, 'allowlist': cold_allowlist}, os.path.join(PROJECT_ROOT, 'artifacts/v1.0/model.joblib'))

with open(os.path.join(PROJECT_ROOT, 'artifacts/v1.0/label_thresholds.json'), 'w') as f:
    json.dump(thresholds, f, indent=2)

metrics_dict = {
    'selected_model': selected_model_name,
    'feature_set': 'cold',
    'selection_metric': 'val_RMSE',
    'validation_metrics': {
        'RMSE': float(primary_res[primary_res['model'] == selected_model_name]['val_RMSE'].iloc[0]),
        'MAE': float(primary_res[primary_res['model'] == selected_model_name]['val_MAE'].iloc[0]),
        'R2': float(primary_res[primary_res['model'] == selected_model_name]['val_R2'].iloc[0]),
        'MAE_orig': float(primary_res[primary_res['model'] == selected_model_name]['val_MAE_orig'].iloc[0])
    },
    'final_test_metrics': {
        'RMSE': float(test_metrics['RMSE']),
        'MAE': float(test_metrics['MAE']),
        'R2': float(test_metrics['R2']),
        'MAE_orig': float(test_metrics['MAE_orig'])
    },
    'robustness_time_split': {
        'val_RMSE': float(res_xgb_t['RMSE']),
        'val_MAE': float(res_xgb_t['MAE']),
        'val_R2': float(res_xgb_t['R2']),
        'val_MAE_orig': float(res_xgb_t['MAE_orig'])
    },
    'classification_metrics': {
        'accuracy': float(acc),
        'majority_baseline_accuracy': float(maj_acc),
        'confusion_matrix': conf_mat.tolist()
    },
    'permutation_importance_top5': imp_df.head(5).to_dict(orient='records'),
    'training_configuration': {
        'seed': seed,
        'train_rows': train_rows
    },
    'limitations': "R^2 is modest due to engagement noise. Models trained on historical data up to May 2019. Follower counts are scrape-time not post-time."
}

with open(os.path.join(PROJECT_ROOT, 'artifacts/v1.0/metrics.json'), 'w') as f:
    json.dump(metrics_dict, f, indent=2)

print("[Phase 6] COMPLETE")


[Phase 6] Saving artifacts...
[Phase 6] COMPLETE
